# IOGraphProcessor demo

This notebook demonstrates [`IOGraphProcessor`](../../src/phopymnehelper/iograph_data.py) from `phopymnehelper`.

IOGraphica exports mouse-position CSV files whose filenames encode approximate session start/end times. The processor:

1. Scans a directory of CSV exports and parses filename time ranges
2. Merges overlapping exports into sessions (incremental saves deduplicated)
3. Builds a single `master_df` with absolute `sample_time` timestamps
4. Exposes timeline-ready `detail_df` (unix `t`, `x`, `y`) and `intervals_df` views

In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
from datetime import datetime
from pathlib import Path

import pandas as pd

from phopymnehelper.iograph_data import IOGraphProcessor

Automatic pdb calling has been turned OFF
Using matplotlib as 2D backend.


In [2]:
# Point this at a folder of IOGraphica CSV exports.
iograph_directory = Path(r"C:\Users\pho\Desktop\IOGraphOutputs").resolve()
assert iograph_directory.is_dir(), f"IOGraph directory not found: {iograph_directory}"

timestamp_str = datetime.now().strftime("%Y-%m-%d_%I%M%p").lower()
output_dir = iograph_directory.parent

## One-shot load: `from_directory`

The simplest entry point scans the folder, merges sessions, and populates `file_table_df` and `master_df`.

In [3]:
processor = IOGraphProcessor.from_directory(iograph_directory, recursive=False, drop_na_coords=True)

print(f"Scanned {len(processor.file_table_df):,} CSV file(s)")
print(f"Built master_df with {len(processor.master_df):,} sample row(s)")
print(f"Sessions: {processor.master_df['session_index'].nunique() if not processor.master_df.empty else 0}")

Scanned 177 CSV file(s)
Built master_df with 7,443,143 sample row(s)
Sessions: 138


In [4]:
processor.file_table_df

,file_path,file_name,created_datetime,modified_datetime,size_bytes,parsed_filename_dt_start,parsed_filename_dt_end
0,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 1 hour (from 0-12 to 1-19).csv,2026-06-21 13:03:05.519413248,2026-06-10 01:19:17.868640512,1070787,2026-06-10 00:12:00,2026-06-10 01:19:00
1,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 1 hour (from 14-37 to 15-40).csv,2026-05-27 11:32:17.199094528,2026-05-12 15:40:17.925492736,151573,2026-05-12 14:37:00,2026-05-12 15:40:00
2,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 1 hour (from 15-08 to 16-14).csv,2026-07-06 16:14:26.657165568,2026-07-06 16:14:26.685317632,1072667,2026-07-06 15:08:00,2026-07-06 16:14:00
3,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 1 hour (from 16-04 to 17-05).csv,2026-06-21 13:03:05.581413376,2026-06-10 17:05:53.737403136,1017543,2026-06-10 16:04:00,2026-06-10 17:05:00
4,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 1 hour (from 17-54 to 18-54).csv,2026-06-24 18:54:46.543892736,2026-06-24 18:54:46.571895296,998005,2026-06-24 17:54:00,2026-06-24 18:54:00
...,...,...,...,...,...,...,...
172,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 8.9 hours (from 6-01 to 15-01).csv,2026-07-02 15:01:53.470700544,2026-07-02 15:01:53.499690496,94079,2026-07-02 06:01:00,2026-07-02 15:01:00
173,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 9 hours (from 23-35 Apr 25th to 8...,2026-05-27 11:32:27.971300608,2026-04-26 08:44:57.151268608,566057,2026-04-25 23:35:00,2026-04-26 08:44:00
174,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphica...,IOGraphica - 9.1 hours (from 0-55 to 10-03).csv,2026-05-27 11:32:27.986271744,2025-08-29 10:04:00.081724416,1564109,2025-08-29 00:55:00,2025-08-29 10:03:00
175,C:\Users\pho\Desktop\IOGraphOutputs\IOGraphOut...,IOGraphOutputs_file_table.csv,2026-06-21 13:03:03.708523520,2026-05-27 19:41:20.361897728,27653,NaT,NaT


In [5]:
master_df = processor.master_df

if not master_df.empty:
    end_check = master_df.groupby("session_index").agg(
        sample_time_max=("sample_time", "max"),
        parsed_filename_dt_end=("parsed_filename_dt_end", "max"),
    )
    end_check["end_delta_sec"] = (
        end_check["sample_time_max"] - end_check["parsed_filename_dt_end"]
    ).dt.total_seconds()
    assert end_check["end_delta_sec"].abs().max() < 1e-3
    assert master_df["session_index"].is_monotonic_increasing
    assert not master_df.duplicated(subset=["session_index", "sample_time"]).any()

master_df.head()

,session_index,sample_time,time,x,y,source_file_name,parsed_filename_dt_start,parsed_filename_dt_end
0,0,2025-05-06 16:52:29.898,0,22.0,-1618.0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv,2025-05-06 16:52:00,2025-05-06 16:58:00
1,0,2025-05-06 16:52:30.343,445,22.0,-1617.0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv,2025-05-06 16:52:00,2025-05-06 16:58:00
2,0,2025-05-06 16:52:30.377,479,24.0,-1614.0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv,2025-05-06 16:52:00,2025-05-06 16:58:00
3,0,2025-05-06 16:52:30.411,513,29.0,-1608.0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv,2025-05-06 16:52:00,2025-05-06 16:58:00
4,0,2025-05-06 16:52:30.445,547,36.0,-1599.0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv,2025-05-06 16:52:00,2025-05-06 16:58:00


In [ ]:
new_cols_df, intervals_df = IOGraphProcessor.compute_psychomotor_metrics(detail_df=processor.detail_df)
intervals_df

## 11m 54s -- len(intervals_df): 107919 rows, len(processor.detail_df): 7443143, len(new_cols_df): 7443143

,t_start,t_duration,backtrack_severity,impairment_metric,grab_failed_event
0,1.746565e+09,0.960,4.795383,0.047954,1.0
1,1.746565e+09,0.549,67.681593,0.676816,0.0
2,1.746565e+09,0.993,17.780065,0.177801,0.0
3,1.746565e+09,0.902,8.650280,0.086503,0.0
4,1.746565e+09,0.926,27.485235,0.274852,0.0
...,...,...,...,...,...
107914,1.783399e+09,0.958,45.289851,0.452899,0.0
107915,1.783399e+09,0.974,6.095781,0.060958,0.0
107916,1.783399e+09,0.976,35.632732,0.356327,0.0
107917,1.783399e+09,0.650,0.063504,0.000635,0.0


In [8]:
new_cols_df

,t,backtrack_severity,impairment_metric,grab_failed_event
0,1.746565e+09,0.0,0.0,0.0
1,1.746565e+09,0.0,0.0,0.0
2,1.746565e+09,0.0,0.0,0.0
3,1.746565e+09,0.0,0.0,0.0
4,1.746565e+09,0.0,0.0,0.0
...,...,...,...,...
7443138,1.783399e+09,0.0,0.0,0.0
7443139,1.783399e+09,0.0,0.0,0.0
7443140,1.783399e+09,0.0,0.0,0.0
7443141,1.783399e+09,0.0,0.0,0.0


In [ ]:
master_df: pd.DataFrame = processor.master_df
master_df

In [9]:
# len(processor.detail_df)
detail_df: pd.DataFrame = processor.detail_df
detail_df


,t,x,y,session_index,source_file_name
0,1.746565e+09,22.0,-1618.0,0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv
1,1.746565e+09,22.0,-1617.0,0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv
2,1.746565e+09,24.0,-1614.0,0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv
3,1.746565e+09,29.0,-1608.0,0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv
4,1.746565e+09,36.0,-1599.0,0,IOGraphica - 5 minutes (from 16-52 to 16-58).csv
...,...,...,...,...,...
7443138,1.783399e+09,3550.0,2132.0,137,IOGraphica - 1.4 hours (from 23-12 Jul 6th to ...
7443139,1.783399e+09,3561.0,2124.0,137,IOGraphica - 1.4 hours (from 23-12 Jul 6th to ...
7443140,1.783399e+09,3562.0,2124.0,137,IOGraphica - 1.4 hours (from 23-12 Jul 6th to ...
7443141,1.783399e+09,3563.0,2131.0,137,IOGraphica - 1.4 hours (from 23-12 Jul 6th to ...


## Timeline views

`detail_df` and `intervals_df` are derived from `master_df` and match what `IOGraphRecordingsTrackDatasource` uses in pyPhoTimeline.

In [ ]:
detail_df = processor.detail_df
intervals_df = processor.intervals_df

print("detail_df columns:", list(detail_df.columns))
print("intervals_df columns:", list(intervals_df.columns))

display(detail_df.head())
display(intervals_df)

## Step-by-step API

You can also call the class methods directly when you need more control (for example, to write intermediate CSVs).

In [ ]:
file_table_path = output_dir / f"IOGraphOutputs_file_table_{timestamp_str}.csv"
master_path = output_dir / f"IOGraphOutputs_master_{timestamp_str}.csv"

file_table_df, _ = IOGraphProcessor.scan_csv_directory(
    iograph_directory,
    output_csv=file_table_path,
    recursive=False,
)
master_df_stepwise, _ = IOGraphProcessor.build_master_df(
    file_table_df,
    output_csv=master_path,
    drop_na_coords=True,
)

stepwise_processor = IOGraphProcessor(
    directory=iograph_directory,
    file_table_df=file_table_df,
    master_df=master_df_stepwise,
)

print(f"Wrote file table to {file_table_path}")
print(f"Wrote master df to {master_path}")
pd.testing.assert_frame_equal(
    processor.master_df.reset_index(drop=True),
    stepwise_processor.master_df.reset_index(drop=True),
)
stepwise_processor.master_df.tail()

## Save from an existing processor instance

In [ ]:
saved_file_table = processor.save_file_table_csv(output_dir / f"IOGraphOutputs_file_table_saved_{timestamp_str}.csv")
saved_master = processor.save_master_csv(output_dir / f"IOGraphOutputs_master_saved_{timestamp_str}.csv")

print(saved_file_table)
print(saved_master)

## Filename time-range parsing

IOGraphica encodes session bounds in filenames. The processor parses both same-day and cross-midnight formats.

In [ ]:
examples = [
    (
        "IOGraphica - 1 hour (from 14-37 to 15-40).csv",
        pd.Timestamp("2026-05-12 19:40:17"),
    ),
    (
        "IOGraphica - 1.3 hours (from 22-56 May 11th to 0-16 May 12th).csv",
        pd.Timestamp("2026-05-12 04:16:17"),
    ),
]

for file_name, modified in examples:
    dt_start, dt_end = IOGraphProcessor._parsed_filename_dt_range(file_name, modified)
    print(f"{file_name}")
    print(f"  start: {dt_start}")
    print(f"  end:   {dt_end}")
    print()